# Using Pre-Trained Model

In [ ]:
!pip install -q transformers sentencepiece nltk rouge-score

from transformers import MarianMTModel, MarianTokenizer
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

english_sentences = [
    "hello", "how are you", "i am fine", "what is your name", "nice to meet you",
    "i love machine learning", "do you like pizza", "good morning", "thank you", "see you later"
]
french_sentences = [
    "bonjour", "comment ça va", "je vais bien", "quel est ton nom", "ravi de vous rencontrer",
    "j'aime l'apprentissage automatique", "aimes-tu la pizza", "bonjour", "merci", "à plus tard"
]

model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

results = []
for i, src in enumerate(english_sentences):
    inputs = tokenizer(src, return_tensors="pt", padding=True)
    translated = model.generate(**inputs)
    tgt = tokenizer.decode(translated[0], skip_special_tokens=True)

    reference = [nltk.word_tokenize(french_sentences[i])]
    prediction = nltk.word_tokenize(tgt)
    bleu = sentence_bleu(reference, prediction)
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2'], use_stemmer=True).score(
        ' '.join(reference[0]), ' '.join(prediction))

    results.append({
        "English": src,
        "French (Reference)": french_sentences[i],
        "Predicted French": tgt,
        "BLEU": round(bleu, 4),
        "ROUGE-1": round(rouge['rouge1'].fmeasure, 4),
        "ROUGE-2": round(rouge['rouge2'].fmeasure, 4)
    })

import pandas as pd
pd.DataFrame(results)

# Using model from scratch with small datset

In [ ]:
# Step 1: Install and import libraries
!pip install -q datasets nltk rouge-score

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import pandas as pd

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# Step 2: Prepare dataset

english_sentences = [
    "hello", "how are you", "i am fine", "what is your name", "nice to meet you",
    "i love machine learning", "do you like pizza", "good morning", "thank you", "see you later"
]

# CRITICAL: wrap every French sentence with <sos> ... <eos>
# Without these, the decoder has no start signal and no stop signal.
french_sentences = [
    "<sos> bonjour <eos>",
    "<sos> comment ça va <eos>",
    "<sos> je vais bien <eos>",
    "<sos> quel est ton nom <eos>",
    "<sos> ravi de vous rencontrer <eos>",
    "<sos> j'aime l'apprentissage automatique <eos>",
    "<sos> aimes-tu la pizza <eos>",
    "<sos> bonjour <eos>",
    "<sos> merci <eos>",
    "<sos> à plus tard <eos>"
]

print(f"Dataset size: {len(english_sentences)} sentence pairs")
print(f"Example: '{english_sentences[1]}' → '{french_sentences[1]}'")

In [ ]:
# Step 3: Tokenization
# Using Keras Tokenizer with oov_token so unseen words get index 1

def tokenize(sentences, num_words=10000):
    tokenizer = Tokenizer(num_words=num_words, oov_token='<OOV>', filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n')  # keep < > for sos/eos
    tokenizer.fit_on_texts(sentences)
    tensor = tokenizer.texts_to_sequences(sentences)
    return tokenizer, pad_sequences(tensor, padding='post')

en_tokenizer, en_tensor = tokenize(english_sentences)
fr_tokenizer, fr_tensor = tokenize(french_sentences)

input_vocab_size = len(en_tokenizer.word_index) + 1
target_vocab_size = len(fr_tokenizer.word_index) + 1

# Verify <sos> and <eos> are in the French vocabulary
sos_id = fr_tokenizer.word_index.get('<sos>')
eos_id = fr_tokenizer.word_index.get('<eos>')
print(f"Input vocab size:  {input_vocab_size}")
print(f"Target vocab size: {target_vocab_size}")
print(f"<sos> token ID: {sos_id}")
print(f"<eos> token ID: {eos_id}")
print(f"Encoder input shape:  {en_tensor.shape}")
print(f"Decoder target shape: {fr_tensor.shape}")
print(f"\nSample encoded French: {fr_tensor[1]}")

assert sos_id is not None, "ERROR: <sos> not found in vocab"
assert eos_id is not None, "ERROR: <eos> not found in vocab"

In [ ]:
# Step 4: Define a PROPER Transformer model
# causal mask, layer norm, dropout, correct cross-attention

class Transformer(tf.keras.Model):
    def __init__(self, input_vocab, target_vocab, d_model=64, num_heads=4,
                 dff=128, pe_input=50, pe_target=50, dropout_rate=0.1):
        super().__init__()
        self.d_model = d_model

        # Embeddings
        self.encoder_embedding = tf.keras.layers.Embedding(input_vocab, d_model)
        self.decoder_embedding = tf.keras.layers.Embedding(target_vocab, d_model)

        # Positional encodings
        self.pos_encoding_input = self.positional_encoding(pe_input, d_model)
        self.pos_encoding_target = self.positional_encoding(pe_target, d_model)

        # Encoder layers
        self.enc_self_attn = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
        self.enc_norm1 = tf.keras.layers.LayerNormalization()
        self.enc_ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),
            tf.keras.layers.Dense(d_model)
        ])
        self.enc_norm2 = tf.keras.layers.LayerNormalization()

        # Decoder layers
        self.dec_self_attn = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
        self.dec_norm1 = tf.keras.layers.LayerNormalization()
        # Separate cross-attention layer
        self.dec_cross_attn = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
        self.dec_norm2 = tf.keras.layers.LayerNormalization()
        self.dec_ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),
            tf.keras.layers.Dense(d_model)
        ])
        self.dec_norm3 = tf.keras.layers.LayerNormalization()

        self.dropout = tf.keras.layers.Dropout(dropout_rate)
        self.final_dense = tf.keras.layers.Dense(target_vocab)

    def positional_encoding(self, max_len, dm):
        pos = np.arange(max_len)[:, np.newaxis]
        i = np.arange(dm)[np.newaxis, :]
        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(dm))
        angle_rads = pos * angle_rates
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
        return tf.cast(angle_rads[np.newaxis, :, :], dtype=tf.float32)  # (1, max_len, dm)

    def causal_mask(self, seq_len):
        """Prevent decoder from seeing future tokens."""
        # Returns a (seq_len, seq_len) boolean mask: True = attend, False = block
        mask = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        return tf.cast(mask, tf.bool)

    def call(self, inputs, training=False):
        inp, tar = inputs
        seq_len_tar = tf.shape(tar)[1]

        # --- Encoder ---
        enc = self.encoder_embedding(inp) * tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        enc = enc + self.pos_encoding_input[:, :tf.shape(inp)[1], :]
        enc = self.dropout(enc, training=training)

        # Encoder self-attention + Add & Norm
        attn_out = self.enc_self_attn(query=enc, value=enc, key=enc, training=training)
        enc = self.enc_norm1(enc + attn_out)
        ffn_out = self.enc_ffn(enc)
        enc_output = self.enc_norm2(enc + ffn_out)

        # --- Decoder ---
        dec = self.decoder_embedding(tar) * tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        dec = dec + self.pos_encoding_target[:, :seq_len_tar, :]
        dec = self.dropout(dec, training=training)

        # Masked self-attention (causal)
        causal = self.causal_mask(seq_len_tar)
        dec_attn = self.dec_self_attn(query=dec, value=dec, key=dec,
                                      attention_mask=causal, training=training)
        dec = self.dec_norm1(dec + dec_attn)

        # Cross-attention — query from decoder, key/value from encoder
        cross_attn = self.dec_cross_attn(query=dec, value=enc_output, key=enc_output,
                                         training=training)
        dec = self.dec_norm2(dec + cross_attn)

        ffn_out = self.dec_ffn(dec)
        dec = self.dec_norm3(dec + ffn_out)

        return self.final_dense(dec)

print("Transformer model class defined.")

In [ ]:
# Step 5: Compile and train
# More epochs + lower learning rate for this tiny dataset

model = Transformer(
    input_vocab=input_vocab_size,
    target_vocab=target_vocab_size,
    d_model=64,
    num_heads=4,
    dff=128,
    pe_input=en_tensor.shape[1],
    pe_target=fr_tensor.shape[1]
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Teacher forcing: input is fr[:, :-1] (everything except last), target is fr[:, 1:] (shifted right)
decoder_input = fr_tensor[:, :-1]   # starts with <sos>, ends before <eos>
decoder_target = fr_tensor[:, 1:]   # starts after <sos>, ends with <eos>

print(f"Decoder input shape:  {decoder_input.shape}  (teacher forcing input)")
print(f"Decoder target shape: {decoder_target.shape}  (what we predict)")
print(f"\nTraining...")

history = model.fit(
    [en_tensor, decoder_input],
    decoder_target,
    epochs=200,
    batch_size=10,
    verbose=0  # suppress per-epoch logs
)

print(f"Final loss:     {history.history['loss'][-1]:.4f}")
print(f"Final accuracy: {history.history['accuracy'][-1]:.4f}")

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'])
ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax2.plot(history.history['accuracy'])
ax2.set_title('Training Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
plt.tight_layout()
plt.show()

In [ ]:
# Step 6: Translation function — Correct autoregressive decoding

def translate(sentence):
    """Translate an English sentence to French using autoregressive decoding."""
    # Encode input
    seq = en_tokenizer.texts_to_sequences([sentence])
    padded = pad_sequences(seq, maxlen=en_tensor.shape[1], padding='post')

    # Initialize decoder with <sos>
    max_target_len = fr_tensor.shape[1]
    decoder_input = np.zeros((1, max_target_len - 1), dtype=np.int32)
    decoder_input[0, 0] = fr_tokenizer.word_index['<sos>']

    result_ids = []

    for i in range(1, max_target_len - 1):
        # pass the full decoder_input (not sliced), index correctly
        output = model([padded, decoder_input], training=False)
        # output shape: (1, max_target_len-1, target_vocab_size)
        # We want the prediction at position i-1 (the token after the last one we set)
        pred_id = tf.argmax(output[0, i - 1], axis=-1).numpy()
        decoder_input[0, i] = pred_id
        result_ids.append(pred_id)

        if pred_id == fr_tokenizer.word_index.get('<eos>', -1):
            break

    # Decode token IDs back to words
    words = []
    for idx in result_ids:
        if idx == 0:
            continue
        word = fr_tokenizer.index_word.get(idx, '')
        if word in ('<eos>', '<sos>', '<OOV>'):
            if word == '<eos>':
                break
            continue
        words.append(word)
    return ' '.join(words)

# Quick test
print("Quick test:")
for s in ["hello", "how are you", "thank you"]:
    print(f"  '{s}' → '{translate(s)}'")

In [ ]:
# Step 7: Evaluate with BLEU and ROUGE

smoother = SmoothingFunction().method1  # avoids 0 BLEU for short sentences
results = []

for i in range(len(english_sentences)):
    ref_text = french_sentences[i].replace('<sos>', '').replace('<eos>', '').strip()
    pred_text = translate(english_sentences[i])

    ref_tokens = [nltk.word_tokenize(ref_text)]
    pred_tokens = nltk.word_tokenize(pred_text)

    bleu = sentence_bleu(ref_tokens, pred_tokens, smoothing_function=smoother)
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2'], use_stemmer=True).score(
        ref_text, pred_text
    )

    results.append({
        "English": english_sentences[i],
        "French (Reference)": ref_text,
        "Predicted French": pred_text,
        "BLEU": round(bleu, 4),
        "ROUGE-1": round(rouge['rouge1'].fmeasure, 4),
        "ROUGE-2": round(rouge['rouge2'].fmeasure, 4)
    })

df = pd.DataFrame(results)
print(f"\nAverage BLEU:    {df['BLEU'].mean():.4f}")
print(f"Average ROUGE-1: {df['ROUGE-1'].mean():.4f}")
df

# Using model from scratch with big datset

In [ ]:


# Step 2: Load a real translation dataset (English-French)
data = load_dataset("opus_books", "en-fr", split='train[:10000]')
english_sentences = [f"{x['translation']['en']}" for x in data]
french_sentences = [f"<sos> {x['translation']['fr']} <eos>" for x in data]

